# **walter**

## **Project Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import subprocess
import getpass

IN_COLAB = "google.colab" in sys.modules

REPO_NAME = "walter"
GIT_BRANCH = "main"
REPO_PATH = Path("/content") / REPO_NAME


def get_tokens():
    github_token = os.getenv("GITHUB_TOKEN") or getpass.getpass("GitHub token: ")
    return github_token


def install_core_ml_stack():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "transformers==4.44.2",
            "accelerate==0.33.0",
            "pyarrow",
        ],
        check=True,
    )


def setup_repo(github_token):
    os.chdir("/content")

    repo_url = f"https://{github_token}@github.com/Mango-Cats/{REPO_NAME}.git"

    if REPO_PATH.exists():
        os.chdir(REPO_PATH)
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "reset", "--hard", f"origin/{GIT_BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", repo_url], check=True)
        os.chdir(REPO_PATH)

    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)


if IN_COLAB:
    print("Colab detected")

    github_token = get_tokens()

    install_core_ml_stack()
    setup_repo(github_token)

    os.chdir(REPO_PATH)
    print("Project root:", REPO_PATH)

In [2]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)

Torch: 2.11.0+cu130
Device: cpu


## **Nomenclature and Terminologies**

The dataset $\mathcal{D}_{\text{raw}}$ (represented as `D_raw` in the source code) refers to the raw Philippine human-drug registry, which is freely available as a `.csv` file at [https://verification.fda.gov.ph/drug_productslist.php](https://verification.fda.gov.ph/drug_productslist.php).

The intermediate dataset, $\mathcal{D}_{\text{clean}}$ (`D_clean`), is the result of passing $\mathcal{D}_{\text{raw}}$ through the preprocessing pipeline. 

The final dataset, $\mathcal{D}_{\text{train}}$ (`D_train`), is used to train a weighted sum of similarity measures via a genetic algorithm. It consists of ordered pairs of drugs formed from the cleaned registry, such that every pair $(x, y) \in \mathcal{D}_{\text{clean}} \times \mathcal{D}_{\text{clean}}$. This training dataset is partitioned into two disjoint subsets:

* **$P \subset \mathcal{D}_{\text{train}}$** (`P`) is the set of known positives, consisting of ordered drug pairs that are manually verified as LASA.
* **$U \subset \mathcal{D}_{\text{train}}$** (`U`) is the unlabeled noise set, consisting of randomly paired drugs from $\mathcal{D}_{\text{clean}}$. $U$ acts as the noise class (its true labels are unknown), so it may contain undetected LASA pairs.

Furthermore, $|U| \gg |P|$, $P \cap U = \emptyset$, and $P \cup U = \mathcal{D}_{\text{train}}$.

## **Preprocessing**

The first step is to preprocess (load, validate, clean) the FDA human drug registry dataset to construct our $\mathcal{D}_\text{clean}$ dataset.

Ensure that the FDA human drug registry dataset exists anywhere starting from the root folder and has the same filename defined by `PRIMARY_FNAME`.

The code for this section is located at [`/src/preprocessing.py`](/src/preprocessing.py).

The function `master_maker` is the coordinator function that performs data loading, validation, cleaning, and reporting. 

In [3]:
import pandas as pd
import src.preprocessing as pre

skip = True

if skip:
    filename = Path(pre.CLEANED_FNAME)
    D_clean = pd.read_parquet(filename)
else:
    D_clean = pre.master_maker(sort=True, save=True)

Let's look at the info of the dataset.

In [4]:
D_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22838 entries, 0 to 22837
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Brand Name  22838 non-null  object
dtypes: object(1)
memory usage: 178.6+ KB


Then, the head of the dataset.

In [5]:
D_clean.head()

,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen


Finally, let's look at a slice of 10 entries in the dataset by using the `get_rand_entries()` function.

In [6]:
display(pre.get_rand_entries(df=D_clean, count=10))

,Brand Name
10505,Insulyf-R (Regular)
10506,Insunova-N
10507,Insunova-R
10508,Insvex
10509,Intalopram 10
10510,Intalopram 20
10511,Intaxel
10512,Intelzinc
10513,Inten SEC
10514,Intercid


## **True LASA Pairs**

Now that we have the $\mathcal{D}_\text{clean}$ we can now proceed with constructing $\mathcal{D}_\text{train}$. We will prioritize constructing the subset of true LASA pairs, or the set $P$.

The code for this section is located at [`/src/proposer/`](/src/proposer/).

### **Local Models**

In [ ]:
from src.proposer.local import LocalModel, response
from src.proposer.prompt import construct_user_prompt
from rapidfuzz import process, fuzz
from src.preprocessing import TARGET_COL

model_choice = LocalModel.DEEPSEEK_1_5B
iterations = 1
P_local = []
all_drugs = D_clean[TARGET_COL].to_list()

for i in range(iterations):
    # Random drug in dataset
    sample_drug = D_clean.sample(n=1)['Brand Name'].iloc[0]
    top_matches = process.extract(sample_drug, all_drugs, scorer=fuzz.WRatio, limit=11)
    
    # A simple fuzzy search to limit context window
    candidate_list = [match[0] for match in top_matches if match[0] != sample_drug][:10]
    candidate_str = "\n".join(candidate_list)
    
    # Decoder-only Inference
    user_prompt = construct_user_prompt(sample_drug, candidate_str, 5)
    raw_output = response(user_prompt, model=model_choice, new_toks_len=512)

    if "</think>" in raw_output:
        clean_proposal = raw_output.split("</think>")[-1].strip()
    else:
        clean_proposal = raw_output.strip()

    P_local.append((sample_str, clean_proposal))

for idx, (data, prop) in enumerate(P_local):
    print(f"--- Iteration {idx + 1} ---")
    print(f"Drug Name: {data}")
    print(f"Proposed LASA Pairs: {prop}\n")

Loading tokenizer for ./models/deepseek-r1-distill-qwen-1.5b...


## **Noise Pairs**

Now that we have $P$, we can now complete constructing $\mathcal{D}_\text{train}$ by constructing the set $U$ or the unlabeled noise set.

The code for this section is located at [`/src/noise.py`](/src/noise.py)

In [ ]:
import src.noise as noise
from src.utils import finder

sample_file: Path = finder(fname="sample_true_lasa.csv")
sample_P: pd.DataFrame = pd.read_csv(filepath_or_buffer=sample_file)

lasa_set = noise.get_lasa_set(true_df=sample_P)

assert len(lasa_set) == 20

sample_U = noise.make_noise(fda_df=D_clean, true_df=sample_P, n=3)

sample_U

## **Assembling**

Now that both subsets are complete. Assembling $\mathcal{D}_\text{train}$ is simply a concatenation of $P$ and $U$. 

In [ ]:
...